In [249]:
# 운동 추천 알고리즘에 사용할 라이브러리 불러오기
import pandas as pd
import numpy as np

In [250]:
# 전처리된 추천 후보 운동 영상 데이터 불러오기
workout_videos = pd.read_csv(

    "data/processed/workout_videos_v2_complete.csv"

)

In [251]:
# 추천 알고리즘에 사용할 운동 영상 데이터 및 필수 컬럼 검증
required_columns = {
    "file_nm",
    "title",
    "file_url",
    "video_length",
    "age_group",
    "strength",
    "muscularEndurance",
    "cardiovascularEndurance",
    "flexibility",
    "agility",
    "power",
    "equipment",
    "dose_type"
}

missing_columns = required_columns - set(workout_videos.columns)

assert not missing_columns, f"필수 컬럼 누락: {missing_columns}"
assert len(workout_videos) == 272
assert workout_videos["file_nm"].nunique() == 272
assert workout_videos["dose_type"].isna().sum() == 0
assert set(workout_videos["dose_type"].unique()) <= {
    "reps",
    "hold",
    "timed"
}

print("추천 후보 영상 수:", len(workout_videos))
print("고유 영상 수:", workout_videos["file_nm"].nunique())
print("dose_type 분포:")
print(workout_videos["dose_type"].value_counts())

display(workout_videos.head())

추천 후보 영상 수: 272
고유 영상 수: 272
dose_type 분포:
dose_type
reps     179
hold      67
timed     26
Name: count, dtype: int64


,file_nm,title,file_url,video_length,age_group,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,equipment,dose_type
0,0AUDLJ08S_00311.mp4,앉았다 일어서면서 점프하기,http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,34,공통,0.0,0.0,0.0,0.0,0.5,0.5,NaN,reps
1,0AUDLJ08S_00237.mp4,옆으로 누워 버티기,http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,40,공통,1.0,0.0,0.0,0.0,0.0,0.0,매트,hold
2,0AUDLJ08S_00830.mp4,점프하며 뛰기,http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,44,청소년,0.0,0.0,0.0,0.0,0.0,1.0,NaN,timed
3,0AUDLJ08S_00287.mp4,짐볼에 다리 올리고 엉덩이 들어올리기,http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,45,공통,1.0,0.0,0.0,0.0,0.0,0.0,짐볼,reps
4,0AUDLJ08S_00316.mp4,점프 팔굽혀펴기,http://openapi.kspo.or.kr/web/video/0AUDLJ08S_...,45,공통,0.0,0.0,0.0,0.0,0.5,0.5,NaN,reps


In [252]:
# 추천 알고리즘에서 사용할 6개 체력요인 정의
fitness_columns = [
    "strength",
    "muscularEndurance",
    "cardiovascularEndurance",
    "flexibility",
    "agility",
    "power"
]

In [253]:
# 실제 운동 영상 데이터로 α 후보별 추천 피드백 루프 시뮬레이션

# --------------------------------------------------
# 1. 시뮬레이션 설정
# --------------------------------------------------

alpha_candidates = [0.20, 0.25, 0.30, 0.40]

weekly_frequencies = [1, 2, 3, 7]

simulation_days = 56
lookback_days = 14
half_life = 7


# --------------------------------------------------
# 2. 사용자 유형 정의
#    1 = 양호 / 2 = 보통 / 3 = 개선 필요
# --------------------------------------------------

user_types = {
    "근력약점형": {
        "strength": 3,
        "muscularEndurance": 2,
        "cardiovascularEndurance": 2,
        "flexibility": 2,
        "agility": 1,
        "power": 1
    },

    "심폐약점형": {
        "strength": 2,
        "muscularEndurance": 2,
        "cardiovascularEndurance": 3,
        "flexibility": 2,
        "agility": 1,
        "power": 1
    },

    "유연성약점형": {
        "strength": 2,
        "muscularEndurance": 2,
        "cardiovascularEndurance": 2,
        "flexibility": 3,
        "agility": 1,
        "power": 1
    },

    "복합약점형": {
        "strength": 3,
        "muscularEndurance": 3,
        "cardiovascularEndurance": 3,
        "flexibility": 2,
        "agility": 2,
        "power": 2
    },

    "균형형": {
        "strength": 2,
        "muscularEndurance": 2,
        "cardiovascularEndurance": 2,
        "flexibility": 2,
        "agility": 2,
        "power": 2
    },
    "미측정포함형": {
    "strength": 2,
    "muscularEndurance": None,
    "cardiovascularEndurance": 3,
    "flexibility": 2,
    "agility": None,
    "power": None
}
}


# --------------------------------------------------
# 3. 국민체력100 수준 → 운동 필요도
# --------------------------------------------------

need_mapping = {
    1: 0.25,
    2: 0.50,
    3: 1.00
}


# --------------------------------------------------
# 4. 주간 운동 빈도에 따라 운동일 생성
# --------------------------------------------------

def make_workout_days(weekly_frequency, total_days):

    weekly_patterns = {
        1: [0],
        2: [0, 3],
        3: [0, 2, 4],
        7: list(range(7))
    }

    pattern = weekly_patterns[weekly_frequency]

    workout_days = []

    for week_start in range(0, total_days, 7):

        for day in pattern:

            workout_day = week_start + day

            if workout_day < total_days:
                workout_days.append(workout_day)

    return workout_days


# --------------------------------------------------
# 5. 최근 14일 운동 로그 → 최근 운동 가중치 계산
# --------------------------------------------------

def calculate_recent_exposure(
    logs,
    current_day,
    alpha
):

    exposure = {
        factor: 0.0
        for factor in fitness_columns
    }

    for log in logs:

        days_ago = current_day - log["day"]

        # 현재 추천 시점 이전의 최근 14일 운동만 반영
        if 1 <= days_ago <= lookback_days:

            # 반감기 7일의 시간 감쇠
            time_weight = (
                0.5 ** (days_ago / half_life)
            )

            # 완료 = 1.0 / 미완료 = 0.5
            completion_weight = (
                1.0
                if log["completed"]
                else 0.5
            )

            # 수행한 운동 영상 조회
            video = workout_videos.loc[
                workout_videos["file_nm"]
                == log["videoId"]
            ].iloc[0]

            # 영상의 체력요인 비중을 최근 운동량에 반영
            for factor in fitness_columns:

                video_weight = video[factor]

                exposure[factor] += (
                    video_weight
                    * time_weight
                    * completion_weight
                )

    # α 적용 후 최대 1.0으로 제한
    for factor in fitness_columns:

        exposure[factor] = min(
            alpha * exposure[factor],
            1.0
        )

    return exposure


# --------------------------------------------------
# 6. 체력 상태 + 최근 운동량 → 추천 우선순위 계산
# --------------------------------------------------

def calculate_priority(
    fitness_data,
    recent_exposure
):

    priority = {}

    for factor in fitness_columns:

        level = fitness_data.get(factor)

        # 측정값이 없는 체력요인
        if level is None:

            priority[factor] = (
                1.5
                * (1 - recent_exposure[factor])
            )

        # 측정값이 있는 체력요인
        else:

            need = need_mapping[level]

            priority[factor] = (
                need
                + (1 - recent_exposure[factor])
            )

    return priority


# --------------------------------------------------
# 7. 체력요인 우선순위로 영상별 추천 점수 계산
# --------------------------------------------------

def recommend_video(priority):

    scores = np.zeros(
        len(workout_videos)
    )

    for factor in fitness_columns:

        scores += (
            workout_videos[factor].to_numpy()
            * priority[factor]
        )

    best_index = np.argmax(scores)

    return workout_videos.iloc[best_index]


# --------------------------------------------------
# 8. 사용자 1명의 추천 과정을 반복 시뮬레이션
# --------------------------------------------------

def simulate_user(
    fitness_data,
    weekly_frequency,
    alpha
):

    workout_days = make_workout_days(
        weekly_frequency,
        simulation_days
    )

    logs = []

    saturation_count = 0
    exposure_check_count = 0

    max_exposure_values = []

    recommendation_counts = {
        factor: 0
        for factor in fitness_columns
    }

    # 3등급 체력요인을 약점으로 정의
    weak_factors = [
        factor
        for factor, level
        in fitness_data.items()
        if level == 3
    ]

    weak_recommendation_count = 0

    for current_day in range(
        simulation_days
    ):

        # 운동하지 않는 날
        if current_day not in workout_days:
            continue

        # 최근 운동량 계산
        recent_exposure = (
            calculate_recent_exposure(
                logs,
                current_day,
                alpha
            )
        )

        # 체력요인별 포화 여부 기록
        for factor in fitness_columns:

            exposure_check_count += 1

            if recent_exposure[factor] >= 1.0:
                saturation_count += 1

        # 해당 운동일의 가장 높은 가중치 기록
        max_exposure_values.append(
            max(recent_exposure.values())
        )

        # 현재 추천 우선순위 계산
        priority = calculate_priority(
            fitness_data,
            recent_exposure
        )

        # 가장 점수가 높은 영상 추천
        video = recommend_video(
            priority
        )

        # 추천 영상에서 비중이 가장 큰 체력요인
        video_weights = {
            factor: video[factor]
            for factor in fitness_columns
        }

        main_factor = max(
            video_weights,
            key=video_weights.get
        )

        recommendation_counts[
            main_factor
        ] += 1

        # 추천 영상에 약점 체력요인이 포함되는지 확인
        if weak_factors:

            if any(
                video[factor] > 0
                for factor in weak_factors
            ):
                weak_recommendation_count += 1

        # 추천 영상을 완료했다고 가정하고 로그 저장
        logs.append({
            "day": current_day,
            "videoId": video["file_nm"],
            "completed": True
        })

    total_workouts = len(logs)

    # 포화율
    if exposure_check_count > 0:

        saturation_rate = (
            saturation_count
            / exposure_check_count
        )

    else:
        saturation_rate = 0

    # 약점 종목 추천 비율
    if (
        total_workouts > 0
        and len(weak_factors) > 0
    ):

        weak_ratio = (
            weak_recommendation_count
            / total_workouts
        )

    else:
        weak_ratio = np.nan

    # 시뮬레이션 중 관측된 최대 가중치
    if max_exposure_values:

        max_weight = max(
            max_exposure_values
        )

    else:
        max_weight = 0

    return {
        "total_workouts": total_workouts,
        "saturation_rate": saturation_rate,
        "max_weight": max_weight,
        "weak_ratio": weak_ratio,

        **{
            f"{factor}_count": count
            for factor, count
            in recommendation_counts.items()
        }
    }


# --------------------------------------------------
# 9. α × 사용자 유형 × 운동 빈도 전체 실행
# --------------------------------------------------

simulation_results = []

for alpha in alpha_candidates:

    for user_type, fitness_data in (
        user_types.items()
    ):

        for weekly_frequency in (
            weekly_frequencies
        ):

            result = simulate_user(
                fitness_data,
                weekly_frequency,
                alpha
            )

            simulation_results.append({
                "alpha": alpha,
                "user_type": user_type,
                "weekly_frequency":
                    weekly_frequency,
                **result
            })


simulation_results_df = pd.DataFrame(
    simulation_results
)


# --------------------------------------------------
# 10. α별 핵심 결과 요약
# --------------------------------------------------

alpha_summary = (
    simulation_results_df
    .groupby("alpha")
    .agg(
        saturation_rate=(
            "saturation_rate",
            "mean"
        ),
        average_max_weight=(
            "max_weight",
            "mean"
        ),
        average_weak_ratio=(
            "weak_ratio",
            "mean"
        )
    )
    .reset_index()
)




# 비율을 % 단위로 변환
alpha_summary["saturation_rate"] = (
    alpha_summary["saturation_rate"]
    * 100
).round(2)

alpha_summary["average_weak_ratio"] = (
    alpha_summary["average_weak_ratio"]
    * 100
).round(2)

alpha_summary["average_max_weight"] = (
    alpha_summary["average_max_weight"]
    .round(3)
)

alpha_summary

,alpha,saturation_rate,average_max_weight,average_weak_ratio
0,0.20,0.00,0.397,86.55
1,0.25,0.00,0.464,83.12
2,0.30,0.02,0.529,76.93
3,0.40,1.17,0.615,70.39


In [254]:
simulation_results_df[
    (simulation_results_df["user_type"] == "미측정포함형")
    & (simulation_results_df["alpha"] == 0.30)
]

,alpha,user_type,weekly_frequency,total_workouts,saturation_rate,max_weight,weak_ratio,strength_count,muscularEndurance_count,cardiovascularEndurance_count,flexibility_count,agility_count,power_count
68,0.3,미측정포함형,1,8,0.0,0.225000,1.0000,0,0,8,0,0,0
69,0.3,미측정포함형,2,16,0.0,0.559349,0.8125,0,0,13,0,3,0
70,0.3,미측정포함형,3,24,0.0,0.646036,0.6250,3,0,15,3,3,0
71,0.3,미측정포함형,7,56,0.0,0.962514,0.3750,14,0,21,10,11,0


## 최종 추천 알고리즘

위 α 후보 시뮬레이션은 과거 `EXPOSURE_ALPHA = 0.30`을 선정한 근거를 확인하기 위한 독립 실험입니다. 실험 영역에서 정의한 함수는 시뮬레이션 전용이며, 아래에서 정의하는 최종 추천 알고리즘 함수와 별개입니다.

아래 영역은 최종 전처리된 추천 후보 데이터를 사용하는 실제 운동 루틴 추천 로직입니다.

In [ ]:
# 추천 알고리즘에서 사용할 기본 설정값 정의
fitness_columns = [
    "strength",
    "muscularEndurance",
    "cardiovascularEndurance",
    "flexibility",
    "agility",
    "power"
]

EXPOSURE_ALPHA = 0.30
LOOKBACK_DAYS = 14
HALF_LIFE = 7

MISSING_PRIORITY = 1.50
RECENT_VIDEO_DAYS = 7

# 사용자가 선택한 오늘의 운동량에 따라 추천할 운동 개수 정의
ROUTINE_SIZE = {
    "light": 3,
    "normal": 5,
    "full": 7
}

BODY_GOAL_WEIGHT = {
    "strength": 1.10,
    "muscularEndurance": 1.10,
    "cardiovascularEndurance": 1.20,
    "flexibility": 1.00,
    "agility": 1.00,
    "power": 1.00
}

# 루틴 구성용 체력요인 그룹
# - 근지구력 단독 영상이 없어 근력과 묶고, 민첩성·순발력도 함께 묶음
ROUTINE_GROUPS = {
    "strength_group": ["strength", "muscularEndurance"],
    "agility_power_group": ["agility", "power"],
    "flexibility_group": ["flexibility"]
}

# 루틴 안 반복 감점: 한 그룹을 배정하면 그 운동을 한 번 수행한 것으로 보고
# 노출도 증가분(α × 영상 비중 1.0)만큼 그룹 우선순위를 낮춘다.
# group_cap은 한 그룹 과점을 막는 상한, IN_ROUTINE_DECAY는 그룹 간 혼합을 만드는 장치.
IN_ROUTINE_DECAY = EXPOSURE_ALPHA

# 슬롯 그룹에 해당하는 영상으로 인정하는 최소 비중
MIN_GROUP_WEIGHT = 0.5

# 화면에 보여줄 루틴 순서: 동적 유연성 → 민첩·순발 → 근력 → 정리 스트레칭
SLOT_ORDER = [
    "flexibility_group",
    "agility_power_group",
    "strength_group",
    "cooldown"
]

# 정리운동으로 우선 사용할 정적 스트레칭 동작
COOLDOWN_KEYWORDS = ["스트레칭", "자세", "숙이기", "늘리기"]

# 항상 사용할 수 있는 생활용품 (전처리의 기본 생활도구 + 대체 표기)
BASIC_TOOLS = {
    "매트", "물병", "물통", "의자", "테이블", "소파", "수건", "베개",
    "벽", "막대", "봉", "줄", "물병(밴드)", "수건(낮은 쿠션)", "봉(의자)"
}

# 영상 도구명을 사용자 선택지 이름으로 통일
TOOL_ALIASES = {
    "아령": "덤벨",
    "스탭퍼": "스텝박스",
    "스텝퍼": "스텝박스",
    "박스": "스텝박스",
    "큰공": "공",
    "메디신볼": "공",
    "테니스공": "공",
    "줄사다리": "사다리"
}

# 처방 유형별 기본 운동량 (MVP 정책값, 방향 근거: Garber et al. 2011)
PRESCRIPTION_RULES = {
    "reps":  {"value": "10~15", "unit": "회", "sets": 2},
    "hold":  {"value": "20~30", "unit": "초 유지", "sets": 2},
    "timed": {"value": "30", "unit": "초", "sets": 2}
}
REST_SEC = 20
MINUTES_PER_EXERCISE = 2

In [256]:
# 국민체력100 중첩 데이터에서 체력요인 측정값 추출
def extract_fitness_data(fitness100):
    return (fitness100 or {}).get("fitness") or {}


test_fitness100 = {
    "bodyComposition": {
        "height": 170,
        "weight": 65
    },
    "fitness": {
        "strength": 2,
        "muscularEndurance": None,
        "cardiovascularEndurance": 3,
        "flexibility": 2,
        "agility": None,
        "power": None
    }
}

test_extracted_fitness = extract_fitness_data(
    test_fitness100
)

assert test_extracted_fitness["strength"] == 2
assert (
    test_extracted_fitness["cardiovascularEndurance"]
    == 3
)

test_extracted_fitness

{'strength': 2,
 'muscularEndurance': None,
 'cardiovascularEndurance': 3,
 'flexibility': 2,
 'agility': None,
 'power': None}

In [257]:
# 영상 로그 1건이 체력요인에 기여하는 비중 조회
def get_log_weights(log):

    matched_video = workout_videos[
        workout_videos["file_nm"] == log.get("videoId")
    ]

    if matched_video.empty:
        return None

    return matched_video.iloc[0][fitness_columns].to_dict()


# 최근 14일 운동 로그를 기반으로 체력요인별 최근 운동 노출도 계산
def calculate_recent_exposure(logs, current_date):

    exposure = {factor: 0.0 for factor in fitness_columns}
    current_date = pd.to_datetime(current_date)

    for log in logs:

        days_ago = (current_date - pd.to_datetime(log["date"])).days

        # 추천일 이전 최근 14일 운동만 반영
        if not (1 <= days_ago <= LOOKBACK_DAYS):
            continue

        log_weights = get_log_weights(log)

        if log_weights is None:
            continue

        time_weight = 0.5 ** (days_ago / HALF_LIFE)
        completion_weight = 1.0 if log["completed"] else 0.5

        for factor in fitness_columns:
            exposure[factor] += (
                log_weights[factor] * time_weight * completion_weight
            )

    for factor in fitness_columns:
        exposure[factor] = min(EXPOSURE_ALPHA * exposure[factor], 1.0)

    return exposure

In [258]:
# 체력등급을 추천 우선순위 계산에 사용할 운동 필요도로 변환
def get_fitness_need(level):

    if level is None or isinstance(level, bool):
        return None

    if level == 1:
        return 0.25

    if level == 2:
        return 0.50

    if (
        isinstance(level, (int, float))
        and level >= 3
    ):
        return 1.00

    return None

In [259]:
# 운동 목적에 따라 체력요인별 목적 가중치 계산
def calculate_goal_weight(goal, fitness_data):
    goal_weight = {}

    for factor in fitness_columns:
        level = fitness_data.get(factor)
        need = get_fitness_need(level)

        if goal == "grade":
            goal_weight[factor] = 1.0

        elif goal == "body":
            goal_weight[factor] = BODY_GOAL_WEIGHT[factor]

        elif goal == "general":
            if need is None:
                goal_weight[factor] = 1.0
            else:
                goal_weight[factor] = 1.5 / (need + 1)

        else:
            raise ValueError("지원하지 않는 운동 목적입니다.")

    return goal_weight

In [260]:
# 체력 수준과 최근 운동 노출도를 결합해 체력요인별 최종 추천 우선순위 계산
def calculate_priority(fitness_data, recent_exposure, goal):

    goal_weight = calculate_goal_weight(goal, fitness_data)
    priority = {}

    for factor in fitness_columns:

        level = fitness_data.get(factor)

        need = get_fitness_need(level)

        if need is None:
            base_priority = (
                MISSING_PRIORITY
                * (1 - recent_exposure[factor])
            )

        else:
            base_priority = (
                need
                + (1 - recent_exposure[factor])
            )

        priority[factor] = (
            base_priority
            * goal_weight[factor]
        )

    return priority

In [261]:
# 체력등급별 운동 필요도와 4등급 우선순위 계산 테스트
test_need_results = {
    "1": get_fitness_need(1),
    "2": get_fitness_need(2),
    "3": get_fitness_need(3),
    "4": get_fitness_need(4),
    "5": get_fitness_need(5),
    "6": get_fitness_need(6),
    "None": get_fitness_need(None),
    "string_3": get_fitness_need("3"),
    "True": get_fitness_need(True)
}

assert test_need_results == {
    "1": 0.25,
    "2": 0.50,
    "3": 1.00,
    "4": 1.00,
    "5": 1.00,
    "6": 1.00,
    "None": None,
    "string_3": None,
    "True": None
}

test_fitness_data_grade_4 = {
    factor: None
    for factor in fitness_columns
}
test_fitness_data_grade_4["strength"] = 4

test_zero_exposure = {
    factor: 0.0
    for factor in fitness_columns
}

test_priority_grade_4 = calculate_priority(
    test_fitness_data_grade_4,
    test_zero_exposure,
    "grade"
)

assert test_priority_grade_4["strength"] == 2.0

{
    "need_results": test_need_results,
    "priority_with_grade_4": test_priority_grade_4
}

{'need_results': {'1': 0.25,
  '2': 0.5,
  '3': 1.0,
  '4': 1.0,
  '5': 1.0,
  '6': 1.0,
  'None': None,
  'string_3': None,
  'True': None},
 'priority_with_grade_4': {'strength': 2.0,
  'muscularEndurance': 1.5,
  'cardiovascularEndurance': 1.5,
  'flexibility': 1.5,
  'agility': 1.5,
  'power': 1.5}}

In [262]:
# 운동 영상 데이터에 존재하는 연령 그룹 확인
workout_videos["age_group"].value_counts(dropna=False)

age_group
공통     139
청소년    133
Name: count, dtype: int64

In [263]:
# 청소년·성인 추천 후보의 체력요인별 영상 수 확인
for target_group in ["청소년", "성인"]:

    candidates = workout_videos[
        workout_videos["age_group"].isin(
            ["공통", target_group]
        )
    ]

    print(f"\n[{target_group}] 총 후보: {len(candidates)}개")

    for factor in fitness_columns:
        count = (candidates[factor] > 0).sum()
        print(f"{factor}: {count}개")


[청소년] 총 후보: 272개
strength: 116개
muscularEndurance: 35개
cardiovascularEndurance: 2개
flexibility: 110개
agility: 23개
power: 34개

[성인] 총 후보: 139개
strength: 81개
muscularEndurance: 0개
cardiovascularEndurance: 2개
flexibility: 43개
agility: 13개
power: 13개


In [264]:
# 사용자 나이에 맞는 추천 후보 필터링
# - 성인 전용 영상(3개)이 모두 전처리 조건을 통과하지 못해,
#   성인 사용자는 일반 맨몸·소도구 동작으로 구성된 청소년 영상을 보완 후보로 포함
def filter_by_age(workout_videos, age):

    if age is None:
        raise ValueError("사용자 나이 정보가 필요합니다.")

    if 13 <= age <= 18:
        target_groups = ["공통", "청소년"]

    elif 19 <= age <= 64:
        target_groups = ["공통", "성인", "청소년"]

    else:
        raise ValueError("MVP 대상 연령(13~64세)이 아닙니다.")

    return workout_videos[
        workout_videos["age_group"].isin(target_groups)
    ].copy()


# 사용자가 보유한 도구로 수행 가능한 영상만 추천 후보로 필터링
def normalize_tool(tool):
    tool = str(tool).strip()
    return TOOL_ALIASES.get(tool, tool)


def filter_by_equipment(candidates, owned_tools):

    available_tools = set(BASIC_TOOLS) | {
        normalize_tool(tool) for tool in (owned_tools or [])
    }

    def is_available(equipment):
        # 도구 정보가 없으면 맨몸 운동
        if pd.isna(equipment) or str(equipment).strip() == "":
            return True

        required_tools = {
            normalize_tool(tool)
            for tool in str(equipment).split("|")
            if tool.strip()
        }
        return required_tools <= available_tools

    return candidates[
        candidates["equipment"].apply(is_available)
    ].copy()

In [265]:
# 최근 7일 동안 수행한 운동 영상을 추천 후보에서 제외
def exclude_recent_videos(candidates, logs, current_date):

    current_date = pd.to_datetime(current_date)

    recent_file_names = []

    for log in logs:

        workout_date = pd.to_datetime(log["date"])
        days_ago = (current_date - workout_date).days

        # 추천일 당일을 포함한 최근 7일의 영상 식별자 저장
        video_id = log.get("videoId")
        if (
            video_id is not None
            and 0 <= days_ago <= RECENT_VIDEO_DAYS
        ):
            recent_file_names.append(
                video_id
            )

    filtered_candidates = candidates[
        ~candidates["file_nm"].isin(
            recent_file_names
        )
    ].copy()

    return filtered_candidates

In [266]:
# 후보 영상마다 체력요인 우선순위와 영상 비율을 결합해 추천 점수 계산
def calculate_video_scores(candidates, priority):

    scored_candidates = candidates.copy()

    scored_candidates["recommendation_score"] = 0.0

    for factor in fitness_columns:

        scored_candidates["recommendation_score"] += (
            scored_candidates[factor]
            * priority[factor]
        )

    return scored_candidates

In [267]:
# 최고점 영상 중 최근 수행 이력이 가장 오래된 영상을 우선 선택
def select_best_video(scored_candidates, logs):

    if scored_candidates.empty:
        raise ValueError(
            "추천 가능한 운동 영상이 없습니다."
        )

    max_score = scored_candidates[
        "recommendation_score"
    ].max()

    # 부동소수점 오차를 고려해 최고점 후보 선택
    best_candidates = scored_candidates[
        np.isclose(
            scored_candidates["recommendation_score"],
            max_score
        )
    ].copy()

    # 영상별 가장 최근 수행일 계산
    last_workout_dates = {}

    for log in logs:
        file_nm = log.get("videoId")
        if file_nm is None:
            continue
        workout_date = pd.to_datetime(log["date"])

        if (
            file_nm not in last_workout_dates
            or workout_date > last_workout_dates[file_nm]
        ):
            last_workout_dates[file_nm] = workout_date

    best_candidates["last_workout_date"] = (
        best_candidates["file_nm"]
        .map(last_workout_dates)
    )

    # 한 번도 수행하지 않은 최고점 영상 우선
    never_used = best_candidates[
        best_candidates["last_workout_date"].isna()
    ]

    if not never_used.empty:
        selected_video = never_used.sample(n=1).iloc[0]

    else:
        # 모두 수행했다면 가장 오래전에 수행한 영상 우선
        oldest_date = best_candidates[
            "last_workout_date"
        ].min()

        oldest_candidates = best_candidates[
            best_candidates["last_workout_date"]
            == oldest_date
        ]

        selected_video = oldest_candidates.sample(n=1).iloc[0]

    return selected_video

In [268]:
# 루틴 슬롯별 체력요인 그룹 배정
# - 그룹 우선순위 = 그룹에 속한 체력요인 우선순위 중 최댓값
# - 같은 그룹을 배정할 때마다 IN_ROUTINE_DECAY만큼 낮춰 한 요인 쏠림 방지
# - 한 그룹은 주운동 슬롯의 절반(올림)을 넘지 않음
def assign_routine_slots(priority, main_slot_count, candidates):

    group_priority = {
        group: max(priority[factor] for factor in factors)
        for group, factors in ROUTINE_GROUPS.items()
    }

    # 후보 영상이 없는 그룹은 배정하지 않음
    available_groups = [
        group for group, factors in ROUTINE_GROUPS.items()
        if (candidates[factors].sum(axis=1) >= MIN_GROUP_WEIGHT).any()
    ]

    group_cap = int(np.ceil(main_slot_count / 2))
    assigned_count = {group: 0 for group in ROUTINE_GROUPS}
    slots = []

    for _ in range(main_slot_count):

        adjusted = {
            group: group_priority[group] - IN_ROUTINE_DECAY * assigned_count[group]
            for group in available_groups
            if assigned_count[group] < group_cap
        }

        if not adjusted:
            break

        top_score = max(adjusted.values())
        top_groups = [
            group for group, score in adjusted.items()
            if np.isclose(score, top_score)
        ]

        # 동점이면 무작위 선택 (특정 그룹이 항상 먼저 뽑히는 것 방지)
        selected_group = str(np.random.choice(top_groups))

        assigned_count[selected_group] += 1
        slots.append(selected_group)

    return slots


class RoutineCompositionError(ValueError):
    pass


# 슬롯별 영상 선택과 선택적 정리 스트레칭으로 오늘의 루틴 구성
def compose_workout_routine(candidates, priority, logs, routine_size):

    if routine_size == 3:
        main_slot_count = 3
        include_cooldown = False
    else:
        main_slot_count = routine_size - 1
        include_cooldown = True

    slots = assign_routine_slots(priority, main_slot_count, candidates)

    scored = calculate_video_scores(candidates, priority)
    selected = []
    used_titles = set()

    for slot in slots:

        factors = ROUTINE_GROUPS[slot]

        pool = scored[
            (scored[factors].sum(axis=1) >= MIN_GROUP_WEIGHT)
            & ~scored["title"].isin(used_titles)
        ]

        # 유연성 주운동 슬롯은 정적 스트레칭과 분리
        if slot == "flexibility_group" and (pool["dose_type"] != "hold").any():
            pool = pool[pool["dose_type"] != "hold"]

        if pool.empty:
            continue

        video = select_best_video(pool, logs)
        selected.append((slot, video))
        used_titles.add(video["title"])

    if include_cooldown:
        # 정리 스트레칭: 유연성 유지형 영상 중 스트레칭·요가 자세 동작 우선
        cooldown_pool = scored[
            (scored["flexibility"] == 1)
            & (scored["dose_type"] == "hold")
            & ~scored["title"].isin(used_titles)
        ]
        stretch_pool = cooldown_pool[
            cooldown_pool["title"].str.contains("|".join(COOLDOWN_KEYWORDS))
        ]
        if not stretch_pool.empty:
            cooldown_pool = stretch_pool

        if not cooldown_pool.empty:
            selected.append(
                ("cooldown", select_best_video(cooldown_pool, logs))
            )

    if len(selected) != routine_size:
        raise RoutineCompositionError(
            "조건을 만족하는 운동 영상이 부족합니다."
        )

    # 화면 표시 순서로 정렬
    selected.sort(key=lambda item: SLOT_ORDER.index(item[0]))

    return selected


# 처방 유형별 운동량 문구 생성
def make_prescription(video):

    rule = PRESCRIPTION_RULES[video["dose_type"]]

    return {
        "doseType": video["dose_type"],
        "value": rule["value"],
        "unit": rule["unit"],
        "sets": rule["sets"],
        "restSec": REST_SEC,
        "text": f"{rule['value']}{rule['unit']} × {rule['sets']}세트"
    }

In [269]:
# 사용자 체력정보·운동기록·목적·운동량·보유 도구를 기반으로 운동 루틴 추천
def recommend_workout_routine(
    age,
    fitness_data,
    logs,
    current_date,
    goal,
    routine_level,
    owned_tools=None
):

    if routine_level not in ROUTINE_SIZE:
        raise ValueError("지원하지 않는 운동량입니다.")

    routine_size = ROUTINE_SIZE[routine_level]

    # 1. 최근 운동 노출도 → 체력요인별 우선순위 (목적 가중치 포함)
    recent_exposure = calculate_recent_exposure(logs, current_date)
    priority = calculate_priority(fitness_data, recent_exposure, goal)

    # 2. 연령 → 보유 도구 → 최근 7일 수행 영상 순으로 후보 필터링
    base_candidates = filter_by_equipment(
        filter_by_age(workout_videos, age),
        owned_tools
    )
    candidates = exclude_recent_videos(base_candidates, logs, current_date)

    # 3. 최근 영상 제외 상태로 전체 루틴을 먼저 구성
    try:
        selected = compose_workout_routine(
            candidates,
            priority,
            logs,
            routine_size
        )
    except RoutineCompositionError:
        # 실제 루틴 구성이 불가능한 경우에만 최근 영상 제외를 해제하고 재구성
        selected = compose_workout_routine(
            base_candidates,
            priority,
            logs,
            routine_size
        )

    routine = [
        {
            "order": order,
            "videoId": video["file_nm"],
            "title": video["title"],
            "videoUrl": str(video["file_url"]),
            "slot": slot,
            "prescription": make_prescription(video)
        }
        for order, (slot, video) in enumerate(selected, start=1)
    ]

    return {
        "routine": routine,
        "estimatedMinutes": len(routine) * MINUTES_PER_EXERCISE
    }

In [270]:
# 운동 이력이 없는 신규 사용자의 첫 운동 루틴 추천 테스트
test_fitness = {
    "strength": 2,
    "muscularEndurance": 3,
    "cardiovascularEndurance": 2,
    "flexibility": 1,
    "agility": None,
    "power": None
}

test_logs = []
test_goal = "grade"
test_routine_level = "normal"

test_result = recommend_workout_routine(
    age=25,
    fitness_data=test_fitness,
    logs=test_logs,
    current_date="2026-09-23",
    goal=test_goal,
    routine_level=test_routine_level
)

test_routine = test_result["routine"]
test_video_ids = [item["videoId"] for item in test_routine]

assert len(test_routine) == 5
assert len(set(test_video_ids)) == 5
assert len({item["title"] for item in test_routine}) == 5
assert test_routine[-1]["slot"] == "cooldown"

test_result

{'routine': [{'order': 1,
   'videoId': '0AUDLJ08S_00613.mp4',
   'title': '누워서 막대 잡고 팔 들어올리기',
   'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00613.mp4',
   'slot': 'flexibility_group',
   'prescription': {'doseType': 'timed',
    'value': '30',
    'unit': '초',
    'sets': 2,
    'restSec': 20,
    'text': '30초 × 2세트'}},
  {'order': 2,
   'videoId': '0AUDLJ08S_00837.mp4',
   'title': '양팔 벌려 높이 뛰기',
   'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00837.mp4',
   'slot': 'agility_power_group',
   'prescription': {'doseType': 'reps',
    'value': '10~15',
    'unit': '회',
    'sets': 2,
    'restSec': 20,
    'text': '10~15회 × 2세트'}},
  {'order': 3,
   'videoId': '0AUDLJ08S_00882.mp4',
   'title': '누워 무릎 기울기',
   'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00882.mp4',
   'slot': 'strength_group',
   'prescription': {'doseType': 'reps',
    'value': '10~15',
    'unit': '회',
    'sets': 2,
    'restSec': 20,
    'text': '10~15회 × 2세트'}},
  {'ord

In [271]:
# 첫 추천 루틴의 체력요인 구성과 추천 점수 및 동점 후보 확인
test_recent_exposure = calculate_recent_exposure(
    test_logs,
    "2026-09-23"
)

test_priority = calculate_priority(
    test_fitness,
    test_recent_exposure,
    test_goal
)

test_candidates = filter_by_equipment(
    filter_by_age(workout_videos, 25),
    []
)

test_candidates = exclude_recent_videos(
    test_candidates,
    test_logs,
    "2026-09-23"
)

test_scored = calculate_video_scores(
    test_candidates,
    test_priority
)

selected_file_names = [
    item["videoId"]
    for item in test_result["routine"]
]

selected_videos = test_scored[
    test_scored["file_nm"].isin(selected_file_names)
]

max_score = test_scored["recommendation_score"].max()

top_candidates = test_scored[
    np.isclose(
        test_scored["recommendation_score"],
        max_score
    )
]

print("체력요인별 priority")
print(test_priority)

print("\n추천된 영상")
display(
    selected_videos[
        [
            "file_nm",
            "title",
            "age_group",
            *fitness_columns,
            "recommendation_score"
        ]
    ]
)

print(f"\n최고 추천 점수: {max_score:.4f}")
print(f"최고점 동점 후보 수: {len(top_candidates)}개")

display(
    top_candidates[
        [
            "file_nm",
            "title",
            "age_group",
            *fitness_columns,
            "recommendation_score"
        ]
    ]
)

체력요인별 priority
{'strength': 1.5, 'muscularEndurance': 2.0, 'cardiovascularEndurance': 1.5, 'flexibility': 1.25, 'agility': 1.5, 'power': 1.5}

추천된 영상


,file_nm,title,age_group,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,recommendation_score
42,0AUDLJ08S_00613.mp4,누워서 막대 잡고 팔 들어올리기,공통,0.0,0.0,0.0,1.0,0.0,0.0,1.25
65,0AUDLJ08S_00837.mp4,양팔 벌려 높이 뛰기,청소년,0.0,0.0,0.0,0.0,0.0,1.0,1.50
213,0AUDLJ08S_00882.mp4,누워 무릎 기울기,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
216,0AUDLJ08S_00586.mp4,배 스트레칭,공통,0.0,0.0,0.0,1.0,0.0,0.0,1.25
266,0AUDLJ08S_00880.mp4,윗몸일으키기,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75



최고 추천 점수: 1.7500
최고점 동점 후보 수: 19개


,file_nm,title,age_group,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,recommendation_score
136,0AUDLJ08S_00878.mp4,V자 싸이클,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
146,0AUDLJ08S_00891.mp4,버피 테스트,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
161,0AUDLJ08S_00887.mp4,엎드려 팔다리 교차올리기,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
184,0AUDLJ08S_00888.mp4,팔다리 교차 버티기,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
199,0AUDLJ08S_00886.mp4,연속 사이드 스쿼트,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
201,0AUDLJ08S_00875.mp4,바로서서 상체 숙이기,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
203,0AUDLJ08S_00877.mp4,의자에 앉아 물병 들고 몸통 돌리기,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
204,0AUDLJ08S_00885.mp4,앞굽이 앉았다 일어서기,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
207,0AUDLJ08S_00892.mp4,크런치 싸이클,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75
212,0AUDLJ08S_00897.mp4,다리 넓게 벌려 앉았다 일어서기,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.75


In [272]:
# 최근 근력·근지구력 운동 이력이 쌓였을 때 추천 변화 확인
test_logs_after_workout = [
    {
        "date": "2026-09-22",
        "videoId": "0AUDLJ08S_00878.mp4",
        "completed": True
    },
    {
        "date": "2026-09-20",
        "videoId": "0AUDLJ08S_00891.mp4",
        "completed": True
    },
    {
        "date": "2026-09-18",
        "videoId": "0AUDLJ08S_00903.mp4",
        "completed": True
    }
]

test_exposure_after = calculate_recent_exposure(
    test_logs_after_workout,
    "2026-09-23"
)

test_priority_after = calculate_priority(
    test_fitness,
    test_exposure_after,
    test_goal
)

test_result_after = recommend_workout_routine(
    age=25,
    fitness_data=test_fitness,
    logs=test_logs_after_workout,
    current_date="2026-09-23",
    goal=test_goal,
    routine_level=test_routine_level
)

print("운동 전 priority")
print(test_priority)

print("\n최근 운동 노출도")
print(test_exposure_after)

print("\n운동 후 priority")
print(test_priority_after)

print("\n다음 추천")
print(test_result_after)

운동 전 priority
{'strength': 1.5, 'muscularEndurance': 2.0, 'cardiovascularEndurance': 1.5, 'flexibility': 1.25, 'agility': 1.5, 'power': 1.5}

최근 운동 노출도
{'strength': 0.33873414539019275, 'muscularEndurance': 0.33873414539019275, 'cardiovascularEndurance': 0.0, 'flexibility': 0.0, 'agility': 0.0, 'power': 0.0}

운동 후 priority
{'strength': 1.1612658546098071, 'muscularEndurance': 1.6612658546098071, 'cardiovascularEndurance': 1.5, 'flexibility': 1.25, 'agility': 1.5, 'power': 1.5}

다음 추천
{'routine': [{'order': 1, 'videoId': '0AUDLJ08S_00961.mp4', 'title': '무릎 들어올리기', 'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00961.mp4', 'slot': 'flexibility_group', 'prescription': {'doseType': 'reps', 'value': '10~15', 'unit': '회', 'sets': 2, 'restSec': 20, 'text': '10~15회 × 2세트'}}, {'order': 2, 'videoId': '0AUDLJ08S_00829.mp4', 'title': '무릎 높이들어 뛰기', 'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00829.mp4', 'slot': 'agility_power_group', 'prescription': {'doseType': 'timed', 

In [273]:
# 운동 이력 반영 후 추천 근거와 최근 7일 영상 제외 여부 확인
age_candidates = filter_by_equipment(
    filter_by_age(workout_videos, 25),
    []
)

filtered_candidates = exclude_recent_videos(
    age_candidates,
    test_logs_after_workout,
    "2026-09-23"
)

scored_candidates = calculate_video_scores(
    filtered_candidates,
    test_priority_after
)

selected_file_names = [
    item["videoId"]
    for item in test_result_after["routine"]
]

selected_videos = scored_candidates[
    scored_candidates["file_nm"].isin(selected_file_names)
]

recent_files = [
    log["videoId"]
    for log in test_logs_after_workout
]

print("추천된 영상")
display(
    selected_videos[
        [
            "file_nm",
            "title",
            "age_group",
            *fitness_columns,
            "recommendation_score"
        ]
    ]
)

print("연령·도구 필터 후 후보 수:", len(age_candidates))
print("최근 7일 제외 후 후보 수:", len(filtered_candidates))

print("\n최근 운동 영상의 후보 포함 여부")

for file_nm in recent_files:
    is_in_candidates = (
        file_nm in filtered_candidates["file_nm"].values
    )

    print(
        file_nm,
        "→",
        "포함" if is_in_candidates else "제외"
    )

추천된 영상


,file_nm,title,age_group,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,recommendation_score
34,0AUDLJ08S_00829.mp4,무릎 높이들어 뛰기,청소년,0.0,0.0,0.0,0.0,0.0,1.0,1.500000
69,0AUDLJ08S_00623.mp4,수건 이용하여 장딴지 늘리기,공통,0.0,0.0,0.0,1.0,0.0,0.0,1.250000
182,0AUDLJ08S_00961.mp4,무릎 들어올리기,청소년,0.0,0.0,0.0,1.0,0.0,0.0,1.250000
212,0AUDLJ08S_00897.mp4,다리 넓게 벌려 앉았다 일어서기,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.411266
248,0AUDLJ08S_00884.mp4,앉았다 일어서기-3,청소년,0.5,0.5,0.0,0.0,0.0,0.0,1.411266


연령·도구 필터 후 후보 수: 173
최근 7일 제외 후 후보 수: 171

최근 운동 영상의 후보 포함 여부
0AUDLJ08S_00878.mp4 → 제외
0AUDLJ08S_00891.mp4 → 제외
0AUDLJ08S_00903.mp4 → 제외


In [274]:
# 일부 체력요인이 미측정된 사용자의 추천 우선순위와 추천 결과 검증
partial_fitness = {
    "strength": 2,
    "muscularEndurance": None,
    "cardiovascularEndurance": 3,
    "flexibility": 2,
    "agility": None,
    "power": None
}

partial_logs = []

partial_exposure = calculate_recent_exposure(
    partial_logs,
    "2026-09-23"
)

partial_priority = calculate_priority(
    partial_fitness,
    partial_exposure,
    test_goal
)

partial_result = recommend_workout_routine(
    age=25,
    fitness_data=partial_fitness,
    logs=partial_logs,
    current_date="2026-09-23",
    goal=test_goal,
    routine_level=test_routine_level
)

print("최근 운동 노출도")
print(partial_exposure)

print("\n체력요인별 priority")
print(partial_priority)

print("\n추천 결과")
print(partial_result)

최근 운동 노출도
{'strength': 0.0, 'muscularEndurance': 0.0, 'cardiovascularEndurance': 0.0, 'flexibility': 0.0, 'agility': 0.0, 'power': 0.0}

체력요인별 priority
{'strength': 1.5, 'muscularEndurance': 1.5, 'cardiovascularEndurance': 2.0, 'flexibility': 1.5, 'agility': 1.5, 'power': 1.5}

추천 결과
{'routine': [{'order': 1, 'videoId': '0AUDLJ08S_00608.mp4', 'title': '진자운동', 'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00608.mp4', 'slot': 'flexibility_group', 'prescription': {'doseType': 'timed', 'value': '30', 'unit': '초', 'sets': 2, 'restSec': 20, 'text': '30초 × 2세트'}}, {'order': 2, 'videoId': '0AUDLJ08S_00839.mp4', 'title': '사이드 한발 뛰기', 'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00839.mp4', 'slot': 'agility_power_group', 'prescription': {'doseType': 'reps', 'value': '10~15', 'unit': '회', 'sets': 2, 'restSec': 20, 'text': '10~15회 × 2세트'}}, {'order': 3, 'videoId': '0AUDLJ08S_00875.mp4', 'title': '바로서서 상체 숙이기', 'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00

In [275]:
# 실제 추천된 첫 영상의 추천 점수와 전체 최고 점수를 비교
selected_file = test_result["routine"][0]["videoId"]

test_exposure = calculate_recent_exposure(
    test_logs,
    "2026-09-23"
)

test_priority = calculate_priority(
    test_fitness,
    test_exposure,
    test_goal
)

test_candidates = filter_by_equipment(
    filter_by_age(workout_videos, 25),
    []
)

test_candidates = exclude_recent_videos(
    test_candidates,
    test_logs,
    "2026-09-23"
)

test_scored = calculate_video_scores(
    test_candidates,
    test_priority
)

selected_video = test_scored[
    test_scored["file_nm"] == selected_file
]

selected_score = selected_video[
    "recommendation_score"
].iloc[0]

max_score = test_scored[
    "recommendation_score"
].max()

print("추천 영상")
display(
    selected_video[
        [
            "file_nm",
            "title",
            "strength",
            "muscularEndurance",
            "cardiovascularEndurance",
            "flexibility",
            "agility",
            "power",
            "recommendation_score"
        ]
    ]
)

print("추천 영상 점수:", selected_score)
print("전체 최고 점수:", max_score)
print(
    "전체 최고점 후보 여부:",
    np.isclose(selected_score, max_score)
)

추천 영상


,file_nm,title,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power,recommendation_score
42,0AUDLJ08S_00613.mp4,누워서 막대 잡고 팔 들어올리기,0.0,0.0,0.0,1.0,0.0,0.0,1.25


추천 영상 점수: 1.25
전체 최고 점수: 1.75
전체 최고점 후보 여부: False


In [276]:
# 오늘 운동 전과 다음날 추천 시점의 체력요인별 노출도 변화를 계산
def calculate_weight_adjustment(
    logs,
    current_date
):

    current_date = pd.to_datetime(current_date)

    # previous: 오늘 운동을 수행하기 전, 오늘 추천 시점의 최근 노출도
    previous = calculate_recent_exposure(
        logs,
        current_date
    )

    # next: 오늘 수행 로그가 daysAgo=1로 반영되는 다음날 추천 시점의 노출도
    updated_logs = list(logs)
    next_date = current_date + pd.Timedelta(days=1)
    next_recent_exposure = calculate_recent_exposure(
        updated_logs,
        next_date
    )

    # 개발자 전달용 WeightAdjustment 생성
    adjustment = {}

    for factor in fitness_columns:

        adjustment[factor] = {
            "previous": round(
                previous[factor], 3
            ),
            "delta": round(
                next_recent_exposure[factor]
                - previous[factor],
                3
            ),
            "next": round(
                next_recent_exposure[factor], 3
            )
        }

    return adjustment

In [277]:
# 오늘까지의 운동 로그를 반영해 다음날 운동 루틴과 내부 가중치 변화를 반환
def run_recommendation(
    profile,
    fitness100,
    logs,
    current_date,
    goal,
    routine_level,
    owned_tools=None
):

    age = profile["age"]

    fitness_data = extract_fitness_data(
        fitness100
    )

    # logs에는 오늘 완료한 운동 로그까지 포함
    updated_logs = list(logs)

    weight_adjustment = calculate_weight_adjustment(
        updated_logs,
        current_date
    )

    next_date = pd.to_datetime(current_date) + pd.Timedelta(days=1)

    next_workout = recommend_workout_routine(
        age,
        fitness_data,
        updated_logs,
        next_date,
        goal,
        routine_level,
        owned_tools
    )

    return {
        "nextWorkout": next_workout,
        "weightAdjustment": weight_adjustment
    }


test_profile = {
    "age": 30,
    "sex": "female"
}

test_fitness100 = {
    "bodyComposition": {
        "height": 165,
        "weight": 55
    },
    "fitness": {
        "strength": 2,
        "muscularEndurance": 3,
        "cardiovascularEndurance": 2,
        "flexibility": 1,
        "agility": None,
        "power": None
    }
}

test_logs = []
current_date = "2026-09-23"
test_goal = "grade"
test_routine_level = "normal"
test_owned_tools = []

test_recommendation_result = run_recommendation(
    test_profile,
    test_fitness100,
    test_logs,
    current_date,
    test_goal,
    test_routine_level,
    test_owned_tools
)

assert set(test_recommendation_result) == {
    "nextWorkout",
    "weightAdjustment"
}
assert set(test_recommendation_result["nextWorkout"]) == {
    "routine",
    "estimatedMinutes"
}
assert len(
    test_recommendation_result["nextWorkout"]["routine"]
) == 5
assert set(
    test_recommendation_result["weightAdjustment"]
) == set(fitness_columns)

for adjustment in test_recommendation_result[
    "weightAdjustment"
].values():
    assert set(adjustment) == {
        "previous",
        "delta",
        "next"
    }

test_recommendation_result

{'nextWorkout': {'routine': [{'order': 1,
    'videoId': '0AUDLJ08S_00918.mp4',
    'title': '고양이 자세',
    'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00918.mp4',
    'slot': 'flexibility_group',
    'prescription': {'doseType': 'reps',
     'value': '10~15',
     'unit': '회',
     'sets': 2,
     'restSec': 20,
     'text': '10~15회 × 2세트'}},
   {'order': 2,
    'videoId': '0AUDLJ08S_00316.mp4',
    'title': '점프 팔굽혀펴기',
    'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00316.mp4',
    'slot': 'agility_power_group',
    'prescription': {'doseType': 'reps',
     'value': '10~15',
     'unit': '회',
     'sets': 2,
     'restSec': 20,
     'text': '10~15회 × 2세트'}},
   {'order': 3,
    'videoId': '0AUDLJ08S_00891.mp4',
    'title': '버피 테스트',
    'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00891.mp4',
    'slot': 'strength_group',
    'prescription': {'doseType': 'reps',
     'value': '10~15',
     'unit': '회',
     'sets': 2,
     'restSec': 20,
  

In [278]:
# 오늘 운동이 없을 때 모든 체력요인의 delta가 0인지 확인
test_logs_no_today = []

test_adjustment = calculate_weight_adjustment(
    logs=test_logs_no_today,
    current_date="2026-09-23"
)

assert all(
    adjustment["delta"] == 0
    for adjustment in test_adjustment.values()
)

test_adjustment

{'strength': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'muscularEndurance': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'cardiovascularEndurance': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'flexibility': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'agility': {'previous': 0.0, 'delta': 0.0, 'next': 0.0},
 'power': {'previous': 0.0, 'delta': 0.0, 'next': 0.0}}

In [279]:
# 동일한 오늘 운동의 완료·미완료에 따른 delta 차이 확인
test_file_nm = test_result["routine"][0]["videoId"]

completed_logs = [
    {
        "date": "2026-09-23",
        "videoId": test_file_nm,
        "completed": True
    }
]

incomplete_logs = [
    {
        "date": "2026-09-23",
        "videoId": test_file_nm,
        "completed": False
    }
]

completed_adjustment = calculate_weight_adjustment(
    logs=completed_logs,
    current_date="2026-09-23"
)

incomplete_adjustment = calculate_weight_adjustment(
    logs=incomplete_logs,
    current_date="2026-09-23"
)

for factor in fitness_columns:
    print(
        factor,
        "| 완료:",
        completed_adjustment[factor]["delta"],
        "| 미완료:",
        incomplete_adjustment[factor]["delta"]
    )

strength | 완료: 0.0 | 미완료: 0.0
muscularEndurance | 완료: 0.0 | 미완료: 0.0
cardiovascularEndurance | 완료: 0.0 | 미완료: 0.0
flexibility | 완료: 0.272 | 미완료: 0.136
agility | 완료: 0.0 | 미완료: 0.0
power | 완료: 0.0 | 미완료: 0.0


In [280]:
# 오늘 서로 다른 운동 2개를 수행했을 때 delta가 합산되는지 확인
strength_video = workout_videos[
    (workout_videos["strength"] > 0)
    & (workout_videos["muscularEndurance"] > 0)
].iloc[0]

cardio_video = workout_videos[
    workout_videos["cardiovascularEndurance"] == 1.0
].iloc[0]

two_workout_logs = [
    {
        "date": "2026-09-23",
        "videoId": strength_video["file_nm"],
        "completed": True
    },
    {
        "date": "2026-09-23",
        "videoId": cardio_video["file_nm"],
        "completed": True
    }
]

two_workout_adjustment = calculate_weight_adjustment(
    logs=two_workout_logs,
    current_date="2026-09-23"
)

print("운동 1:", strength_video["title"])
print("운동 2:", cardio_video["title"])
print()

for factor in fitness_columns:
    print(
        factor,
        "| delta:",
        two_workout_adjustment[factor]["delta"]
    )

운동 1: V자 싸이클
운동 2: 줄넘기

strength | delta: 0.136
muscularEndurance | delta: 0.136
cardiovascularEndurance | delta: 0.272
flexibility | delta: 0.0
agility | delta: 0.0
power | delta: 0.0


In [281]:
# 테스트 ③에 사용한 두 영상의 실제 체력요인 가중치 확인
display(
    workout_videos[
        workout_videos["file_nm"].isin([
            strength_video["file_nm"],
            cardio_video["file_nm"]
        ])
    ][
        [
            "file_nm",
            "title",
            "strength",
            "muscularEndurance",
            "cardiovascularEndurance",
            "flexibility",
            "agility",
            "power"
        ]
    ]
)

,file_nm,title,strength,muscularEndurance,cardiovascularEndurance,flexibility,agility,power
95,0AUDLJ08S_00464.mp4,줄넘기,0.0,0.0,1.0,0.0,0.0,0.0
136,0AUDLJ08S_00878.mp4,V자 싸이클,0.5,0.5,0.0,0.0,0.0,0.0


In [282]:
# 오늘 수행한 영상이 다음날 후보에서 제외되고 내부 가중치에 반영되는지 검증
today_video = test_recommendation_result["nextWorkout"]["routine"][0]["videoId"]

today_logs = [
    {
        "date": "2026-09-23",
        "videoId": today_video,
        "completed": True
    }
]

today_age_candidates = filter_by_equipment(
    filter_by_age(workout_videos, test_profile["age"]),
    test_owned_tools
)
today_candidates = exclude_recent_videos(
    today_age_candidates,
    today_logs,
    "2026-09-24"
)

assert today_video not in today_candidates["file_nm"].values

today_result = run_recommendation(
    test_profile,
    test_fitness100,
    today_logs,
    "2026-09-23",
    test_goal,
    test_routine_level,
    test_owned_tools
)

today_routine = today_result["nextWorkout"]["routine"]
today_video_ids = [item["videoId"] for item in today_routine]

assert len(today_routine) == 5
assert len(set(today_video_ids)) == 5
assert today_video not in today_video_ids

assert any(
    change["delta"] > 0
    for change in today_result["weightAdjustment"].values()
)

expected_next_exposure = calculate_recent_exposure(
    today_logs,
    "2026-09-24"
)
for factor in fitness_columns:
    assert today_result["weightAdjustment"][factor]["next"] == round(
        expected_next_exposure[factor], 3
    )

today_result

{'nextWorkout': {'routine': [{'order': 1,
    'videoId': '0AUDLJ08S_00832.mp4',
    'title': '팔 벌려 뛰기-1',
    'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00832.mp4',
    'slot': 'agility_power_group',
    'prescription': {'doseType': 'reps',
     'value': '10~15',
     'unit': '회',
     'sets': 2,
     'restSec': 20,
     'text': '10~15회 × 2세트'}},
   {'order': 2,
    'videoId': '0AUDLJ08S_00838.mp4',
    'title': '앞차기하며 뛰기',
    'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00838.mp4',
    'slot': 'agility_power_group',
    'prescription': {'doseType': 'timed',
     'value': '30',
     'unit': '초',
     'sets': 2,
     'restSec': 20,
     'text': '30초 × 2세트'}},
   {'order': 3,
    'videoId': '0AUDLJ08S_00889.mp4',
    'title': '몸통 들어올리기-3',
    'videoUrl': 'http://openapi.kspo.or.kr/web/video/0AUDLJ08S_00889.mp4',
    'slot': 'strength_group',
    'prescription': {'doseType': 'reps',
     'value': '10~15',
     'unit': '회',
     'sets': 2,
     'restSec': 20

In [283]:
# 루틴 구성 검증: 정책·중복·처방·그룹 상한·최근 영상 fallback
np.random.seed(42)

test_users = {
    "근력약점형": {"strength": 3, "muscularEndurance": 2, "cardiovascularEndurance": 2,
                "flexibility": 2, "agility": 1, "power": 1},
    "유연성약점형": {"strength": 2, "muscularEndurance": 2, "cardiovascularEndurance": 2,
                 "flexibility": 3, "agility": 1, "power": 1},
    "균형형": {"strength": 2, "muscularEndurance": 2, "cardiovascularEndurance": 2,
             "flexibility": 2, "agility": 2, "power": 2},
    "미측정포함형": {"strength": 2, "muscularEndurance": None, "cardiovascularEndurance": 3,
                "flexibility": 2, "agility": None, "power": None}
}
tool_cases = {"도구 없음": [], "밴드·덤벨": ["밴드", "덤벨"]}

routine_check = []

for user_type, fitness in test_users.items():
    for goal in ["grade", "body", "general"]:
        for routine_level, size in ROUTINE_SIZE.items():
            for tool_label, tools in tool_cases.items():

                result = recommend_workout_routine(
                    age=25, fitness_data=fitness, logs=[],
                    current_date="2026-09-28", goal=goal,
                    routine_level=routine_level, owned_tools=tools
                )
                routine = result["routine"]
                slots = [item["slot"] for item in routine]
                main_slots = [s for s in slots if s != "cooldown"]
                selected = workout_videos.set_index("file_nm").loc[
                    [item["videoId"] for item in routine]
                ]

                # 개수·중복·영상 기준 예상 시간
                assert len(routine) == size
                assert len({item["videoId"] for item in routine}) == size
                assert len({item["title"] for item in routine}) == size
                assert result["estimatedMinutes"] == size * MINUTES_PER_EXERCISE
                assert set(result) == {"routine", "estimatedMinutes"}

                # 처방 유형과 완성된 영상 URL
                for item in routine:
                    video = selected.loc[item["videoId"]]
                    prescription = item["prescription"]
                    assert prescription["doseType"] in {
                        "reps", "hold", "timed"
                    }
                    assert prescription["doseType"] == video["dose_type"]
                    assert item["videoUrl"] == str(video["file_url"])

                # light는 주운동 3개, normal/full은 마지막 1개가 cooldown
                if routine_level == "light":
                    main_slot_count = 3
                    assert "cooldown" not in slots
                    assert len(main_slots) == 3
                else:
                    main_slot_count = size - 1
                    assert slots[-1] == "cooldown"
                    assert len(main_slots) == main_slot_count
                    cooldown = selected.loc[routine[-1]["videoId"]]
                    assert cooldown["flexibility"] == 1
                    assert cooldown["dose_type"] == "hold"

                for item in routine:
                    if item["slot"] == "flexibility_group":
                        assert selected.loc[item["videoId"], "dose_type"] != "hold"

                # 한 그룹이 실제 주운동 슬롯의 절반(올림)을 넘지 않음
                cap = int(np.ceil(main_slot_count / 2))
                assert max(main_slots.count(g) for g in set(main_slots)) <= cap

                # 보유하지 않은 도구 영상이 없음
                assert len(filter_by_equipment(selected.reset_index(), tools)) == size

                routine_check.append({
                    "user_type": user_type, "goal": goal,
                    "routine_level": routine_level, "tools": tool_label,
                    **{g: main_slots.count(g) for g in ROUTINE_GROUPS},
                    "minutes": result["estimatedMinutes"]
                })

# 최근 영상 제외 상태에서 루틴을 완성할 수 있으면 제외 규칙 유지
recent_video_id = test_result["routine"][0]["videoId"]
recent_log = [{
    "date": "2026-09-27",
    "videoId": recent_video_id,
    "completed": True
}]
recent_excluded_result = recommend_workout_routine(
    age=25,
    fitness_data=test_fitness,
    logs=recent_log,
    current_date="2026-09-28",
    goal="grade",
    routine_level="normal",
    owned_tools=[]
)
assert recent_video_id not in {
    item["videoId"]
    for item in recent_excluded_result["routine"]
}

# 최근 영상 제외로 후보가 비면 base_candidates에서 루틴 전체를 다시 구성
fallback_base_candidates = filter_by_equipment(
    filter_by_age(workout_videos, 25),
    []
)
fallback_logs = [
    {
        "date": "2026-09-27",
        "videoId": file_nm,
        "completed": True
    }
    for file_nm in fallback_base_candidates["file_nm"]
]
fallback_result = recommend_workout_routine(
    age=25,
    fitness_data=test_fitness,
    logs=fallback_logs,
    current_date="2026-09-28",
    goal="grade",
    routine_level="full",
    owned_tools=[]
)
fallback_video_ids = {
    item["videoId"]
    for item in fallback_result["routine"]
}
assert len(fallback_result["routine"]) == 7
assert fallback_video_ids <= set(fallback_base_candidates["file_nm"])

routine_check_df = pd.DataFrame(routine_check)
print("검증 통과 조합 수:", len(routine_check_df))
print("최근 영상 제외 유지 및 후보 부족 fallback 검증 통과")
display(
    routine_check_df[routine_check_df["tools"] == "도구 없음"]
    .pivot_table(
        index=["user_type", "goal"], columns="routine_level",
        values=["strength_group", "agility_power_group", "flexibility_group"]
    )
)

검증 통과 조합 수: 72
최근 영상 제외 유지 및 후보 부족 fallback 검증 통과


agility_power_group              flexibility_group        \
routine_level                    full light normal              full light   
user_type goal                                                               
균형형       body                    2.0   1.0    1.0               2.0   1.0   
          general                 2.0   1.0    1.0               2.0   1.0   
          grade                   2.0   1.0    1.0               2.0   1.0   
근력약점형     body                    1.0   0.0    1.0               2.0   1.0   
          general                 2.0   1.0    2.0               2.0   1.0   
          grade                   1.0   0.0    1.0               2.0   1.0   
미측정포함형    body                    2.0   1.0    1.0               2.0   1.0   
          general                 2.0   1.0    1.0               2.0   1.0   
          grade                   2.0   1.0    2.0               2.0   1.0   
유연성약점형    body                    1.0   0.0    0.0               3.0   2.0   
          general                 2.0   1.0    1.0               2.0   1.0   
          grade                   1.0   0.0    1.0               3.0   2.0   

                         strength_group               
routine_level     normal           full light normal  
user_type goal                                        
균형형       body       1.0            2.0   1.0    2.0  
          general    2.0            2.0   1.0    1.0  
          grade      1.0            2.0   1.0    2.0  
근력약점형     body       1.0            3.0   2.0    2.0  
          general    1.0            2.0   1.0    1.0  
          grade      1.0            3.0   2.0    2.0  
미측정포함형    body       1.0            2.0   1.0    2.0  
          general    1.0            2.0   1.0    2.0  
          grade      1.0            2.0   1.0    1.0  
유연성약점형    body       2.0            2.0   1.0    2.0  
          general    2.0            2.0   1.0    1.0  
          grade      2.0            2.0   1.0    1.0